# Notebook 02 — Golden Dataset Generation Pipeline (200 Queries)

**Thesis:** RAG-Driven Natural Language Energy Demand Forecasting  
**Author:** Zoheb Anwar Hussain (Student ID: 1196931)  
**Model:** Gemini 2.5 Flash (`gemini-2.5-flash`)

---

### What this notebook does

Generates a **200-query evaluation dataset** with Gemini 2.5 Flash reference answers
grounded in KB context. The golden dataset is the ground truth used to evaluate
all experiments (EXP_01 to EXP_10) via RAGAS metrics.

**Phase 1 — Original 50 queries** (GEFCom: 20, Household: 18, Cross-scale: 12)  
Sourced from `src/golden_dataset/query_bank.py`.

**Phase 2 — Extension 150 queries** (GEFCom: 60, Household: 54, Cross-scale: 36)  
Sourced from `src/golden_dataset/query_bank_extended.py`.  
Same type and difficulty proportions as Phase 1.  
Two extra columns added: `citation_required` and `difficulty_aware` for Novelty 1/2 routing.

**Phase 3 — Merge** → `combined_golden_dataset_200.csv`  
Clean sequential integer `golden_id`s (1 to 200).  
This file is what `08_experiments.ipynb` loads.

### Resume safety
Every `generate_golden_dataset()` call is idempotent — already-completed `golden_id`s  
are detected and skipped automatically. Safe to re-run after any interruption.

### Estimated runtime
- Phase 1: ~8 min (50 queries at 4.5s delay + Gemini processing)  
- Phase 2: ~23 min (150 queries at 4.5s delay + Gemini processing)  
- Phase 3: instant  
- **Total: ~31 minutes**

---
## Cell 1 — Imports and Setup

In [1]:
%load_ext autoreload
%autoreload 2

# ── Configuration ────────────────────────────────────────────────────────────
from config import PATHS, GOLDEN_MODEL_NAME
from config.paths import create_all_directories

# ── Utilities ────────────────────────────────────────────────────────────────
from src.utils import setup_logger, log_section

# ── Golden dataset pipeline ───────────────────────────────────────────────────
from src.golden_dataset import (
    load_kb_summaries,
    select_kb_context,
    select_cross_scale_context,
    configure_gemini_golden,
    generate_golden_dataset,
    build_combined_golden_dataset,
    GEFCOM_QUERIES,
    HOUSEHOLD_QUERIES,
    CROSS_SCALE_QUERIES,
)

from src.golden_dataset.query_bank import (
    GEFCOM_QUERIES_EXT,
    HOUSEHOLD_QUERIES_EXT,
    CROSS_SCALE_QUERIES_EXT,
)

import pandas as pd

logger = setup_logger("golden_dataset")
create_all_directories()

logger.info("All imports successful.")
logger.info("Model: %s", GOLDEN_MODEL_NAME)

print("Phase 1 — Original queries:")
print(f"  GEFCom:      {len(GEFCOM_QUERIES)}")
print(f"  Household:   {len(HOUSEHOLD_QUERIES)}")
print(f"  Cross-scale: {len(CROSS_SCALE_QUERIES)}")
print(f"  Subtotal:    {len(GEFCOM_QUERIES)+len(HOUSEHOLD_QUERIES)+len(CROSS_SCALE_QUERIES)}")
print()
print("Phase 2 — Extension queries:")
print(f"  GEFCom:      {len(GEFCOM_QUERIES_EXT)}")
print(f"  Household:   {len(HOUSEHOLD_QUERIES_EXT)}")
print(f"  Cross-scale: {len(CROSS_SCALE_QUERIES_EXT)}")
print(f"  Subtotal:    {len(GEFCOM_QUERIES_EXT)+len(HOUSEHOLD_QUERIES_EXT)+len(CROSS_SCALE_QUERIES_EXT)}")
print()
print(f"Total after merge: {len(GEFCOM_QUERIES)+len(HOUSEHOLD_QUERIES)+len(CROSS_SCALE_QUERIES)+len(GEFCOM_QUERIES_EXT)+len(HOUSEHOLD_QUERIES_EXT)+len(CROSS_SCALE_QUERIES_EXT)}")

2026-05-05 22:57:51 | INFO     | golden_dataset | All imports successful.
2026-05-05 22:57:51 | INFO     | golden_dataset | Model: gemini-2.5-flash


Phase 1 — Original queries:
  GEFCom:      20
  Household:   18
  Cross-scale: 12
  Subtotal:    50

Phase 2 — Extension queries:
  GEFCom:      60
  Household:   54
  Cross-scale: 36
  Subtotal:    150

Total after merge: 200


---
## Cell 2 — Configure Gemini Client

Uses `gemini-2.5-flash` — different generation from the KB model (`gemini-2.0-flash-preview`).  
Using a different model reduces style-matching contamination in faithfulness evaluation.

In [2]:
gemini_golden_client = configure_gemini_golden()
print(f"Gemini client configured. Model: {GOLDEN_MODEL_NAME}")

Gemini client configured. Model: gemini-2.5-flash


---
## Cell 3 — Load Knowledge Base Summaries

Loads `combined_master_summaries.csv` produced by `notebooks/01_kb_generation.ipynb`.

In [3]:
log_section("Loading Knowledge Base summaries", 1, 7)

master_kb    = load_kb_summaries(PATHS["summaries_csv"])
gefcom_kb    = load_kb_summaries(PATHS["summaries_csv"], dataset_filter="gefcom")
household_kb = load_kb_summaries(PATHS["summaries_csv"], dataset_filter="household")

logger.info(
    "KB loaded — GEFCom: %d | Household: %d | Total: %d",
    len(gefcom_kb), len(household_kb), len(master_kb),
)

print(f"KB loaded — GEFCom: {len(gefcom_kb)} | Household: {len(household_kb)} | Total: {len(master_kb)}")
master_kb[["kb_id", "dataset", "granularity", "summary"]].head(3)

2026-05-05 22:58:12 | INFO     | golden_dataset | KB loaded — GEFCom: 105 | Household: 35 | Total: 140



  STEP 1/7  [█░░░░░░] 14%
  Loading Knowledge Base summaries

KB loaded — GEFCom: 105 | Household: 35 | Total: 140


,kb_id,dataset,granularity,summary
0,1,gefcom,daily,"On Sunday, August 7, 2005, Zone 1 recorded a t..."
1,2,gefcom,daily,"On Sunday, August 7, 2005, Zone 3 recorded a t..."
2,3,gefcom,daily,"On Thursday, November 29, 2007, Zone 4 recorde..."


---
## Cell 4 — Phase 1: Generate Original 50 Queries

Generates reference answers for the original 50 queries.  
**Resume-safe:** already-completed `golden_id`s are skipped automatically.  
Estimated: ~8 minutes.

In [4]:
log_section("Phase 1 — Generating original 50 golden queries", 2, 7)

# ── GEFCom golden dataset (20 queries) ───────────────────────────────────────
logger.info("Generating GEFCom golden dataset (20 queries)")
gefcom_golden_df = generate_golden_dataset(
    client              = gemini_golden_client,
    query_list          = GEFCOM_QUERIES,
    kb_df               = gefcom_kb,
    dataset_source      = "gefcom",
    output_path         = PATHS["golden_dataset"] / "gefcom_golden_dataset.csv",
    context_selector_fn = select_kb_context,
)
logger.info("GEFCom golden complete: %d entries.", len(gefcom_golden_df))
print(f"GEFCom: {len(gefcom_golden_df)} queries generated")

# ── Household golden dataset (18 queries) ────────────────────────────────────
logger.info("Generating Household golden dataset (18 queries)")
household_golden_df = generate_golden_dataset(
    client              = gemini_golden_client,
    query_list          = HOUSEHOLD_QUERIES,
    kb_df               = household_kb,
    dataset_source      = "household",
    output_path         = PATHS["golden_dataset"] / "household_golden_dataset.csv",
    context_selector_fn = select_kb_context,
)
logger.info("Household golden complete: %d entries.", len(household_golden_df))
print(f"Household: {len(household_golden_df)} queries generated")

# ── Cross-scale golden dataset (12 queries) ───────────────────────────────────
logger.info("Generating Cross-scale golden dataset (12 queries)")

def cross_scale_selector(query_meta, kb_df):
    return select_cross_scale_context(query_meta, master_kb_df=kb_df)

cross_scale_golden_df = generate_golden_dataset(
    client              = gemini_golden_client,
    query_list          = CROSS_SCALE_QUERIES,
    kb_df               = master_kb,
    dataset_source      = "cross_scale",
    output_path         = PATHS["golden_dataset"] / "cross_scale_golden_dataset.csv",
    context_selector_fn = cross_scale_selector,
)
logger.info("Cross-scale golden complete: %d entries.", len(cross_scale_golden_df))
print(f"Cross-scale: {len(cross_scale_golden_df)} queries generated")

print()
print(f"Phase 1 complete: {len(gefcom_golden_df)+len(household_golden_df)+len(cross_scale_golden_df)} queries total")

2026-05-05 22:58:27 | INFO     | golden_dataset | Generating GEFCom golden dataset (20 queries)



  STEP 2/7  [██░░░░░] 28%
  Phase 1 — Generating original 50 golden queries



Generating gefcom answers: 0it [00:00, ?it/s]

2026-05-05 22:58:27 | INFO     | golden_dataset | GEFCom golden complete: 20 entries.
2026-05-05 22:58:27 | INFO     | golden_dataset | Generating Household golden dataset (18 queries)


GEFCom: 20 queries generated


...

2026-05-05 22:58:27 | INFO     | golden_dataset | Household golden complete: 18 entries.
2026-05-05 22:58:27 | INFO     | golden_dataset | Generating Cross-scale golden dataset (12 queries)


Household: 18 queries generated


Generating cross_scale answers: 0it [00:00, ?it/s]

2026-05-05 22:58:28 | INFO     | golden_dataset | Cross-scale golden complete: 12 entries.


Cross-scale: 12 queries generated

Phase 1 complete: 50 queries total


---
## Cell 5 — Phase 2: Generate Extension 150 Queries

Generates reference answers for the 150 more queries 
Same proportions as Phase 1 (3x scale).  
**Resume-safe:** interrupted runs resume from where they stopped.  
Estimated: ~23 minutes.

In [5]:
log_section("Phase 2 — Generating extension 150 golden queries", 3, 7)

# ── GEFCom extension (60 queries) ────────────────────────────────────────────
logger.info("Generating GEFCom extension (60 queries)")
gefcom_golden_ext_df = generate_golden_dataset(
    client              = gemini_golden_client,
    query_list          = GEFCOM_QUERIES_EXT,
    kb_df               = gefcom_kb,
    dataset_source      = "gefcom_ext",
    output_path         = PATHS["golden_dataset"] / "gefcom_golden_ext.csv",
    context_selector_fn = select_kb_context,
)
logger.info("GEFCom ext complete: %d entries.", len(gefcom_golden_ext_df))
print(f"GEFCom ext: {len(gefcom_golden_ext_df)} queries generated")

# ── Household extension (54 queries) ─────────────────────────────────────────
logger.info("Generating Household extension (54 queries)")
household_golden_ext_df = generate_golden_dataset(
    client              = gemini_golden_client,
    query_list          = HOUSEHOLD_QUERIES_EXT,
    kb_df               = household_kb,
    dataset_source      = "household_ext",
    output_path         = PATHS["golden_dataset"] / "household_golden_ext.csv",
    context_selector_fn = select_kb_context,
)
logger.info("Household ext complete: %d entries.", len(household_golden_ext_df))
print(f"Household ext: {len(household_golden_ext_df)} queries generated")

# ── Cross-scale extension (36 queries) ───────────────────────────────────────
logger.info("Generating Cross-scale extension (36 queries)")
cross_scale_golden_ext_df = generate_golden_dataset(
    client              = gemini_golden_client,
    query_list          = CROSS_SCALE_QUERIES_EXT,
    kb_df               = master_kb,
    dataset_source      = "cross_scale_ext",
    output_path         = PATHS["golden_dataset"] / "cross_scale_golden_ext.csv",
    context_selector_fn = cross_scale_selector,
)
logger.info("Cross-scale ext complete: %d entries.", len(cross_scale_golden_ext_df))
print(f"Cross-scale ext: {len(cross_scale_golden_ext_df)} queries generated")

print()
print(f"Phase 2 complete: {len(gefcom_golden_ext_df)+len(household_golden_ext_df)+len(cross_scale_golden_ext_df)} queries total")

2026-05-05 22:59:10 | INFO     | golden_dataset | Generating GEFCom extension (60 queries)



  STEP 3/7  [███░░░░] 42%
  Phase 2 — Generating extension 150 golden queries



Generating gefcom_ext answers:   0%|          | 0/60 [00:00<?, ?it/s]

Attempt 1/3 failed for 'gefcom_ext_comparative_034': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1/3 failed for 'gefcom_ext_zone_specific_044': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
2026-05-05 23:07:50 | INFO     | golden_dataset | GEFCom ext complete: 60 entries.
2026-05-05 23:07:50 | INFO     | golden_dataset | Generating Household extension (54 queries)


GEFCom ext: 60 queries generated


...

Attempt 1/3 failed for 'household_ext_statistical_001': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1/3 failed for 'household_ext_statistical_004': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1/3 failed for 'household_ext_pattern_010': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1/3 failed for 'household_ext_pattern_015': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Atte

Household ext: 54 queries generated


Generating cross_scale_ext answers:   0%|          | 0/36 [00:00<?, ?it/s]

Attempt 1/3 failed for 'cross_scale_ext_cross_scale_008': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1/3 failed for 'cross_scale_ext_cross_scale_010': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1/3 failed for 'cross_scale_ext_cross_scale_028': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 2/3 failed for 'cross_scale_ext_cross_scale_028': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNA

Cross-scale ext: 36 queries generated

Phase 2 complete: 150 queries total


---
## Cell 6 — Phase 3: Build Combined 200-Query Dataset

Merges all 6 per-source CSVs into `combined_golden_dataset_200.csv`.  
Attaches `citation_required` and `difficulty_aware` columns from the query bank.  
Reassigns clean sequential integer `golden_id`s (1 to 200).

In [6]:
log_section("Phase 3 — Building combined 200-query golden dataset", 4, 7)

from src.golden_dataset.generator import GOLDEN_CSV_COLUMNS

# ── Build novelty-column lookup from extended query bank ─────────────────────
# citation_required: 'yes' = query expected to trigger [En] citations (EXP_05/06/07)
# difficulty_aware:  'yes' = query is a candidate for difficulty scoring (EXP_08/09)
all_ext_queries = GEFCOM_QUERIES_EXT + HOUSEHOLD_QUERIES_EXT + CROSS_SCALE_QUERIES_EXT
novelty_lookup = {
    q["user_query"].strip(): {
        "citation_required": q.get("citation_required", "no"),
        "difficulty_aware":  q.get("difficulty_aware",  "no"),
    }
    for q in all_ext_queries
}

# ── Attach novelty columns to extension DFs ───────────────────────────────────
for df in [gefcom_golden_ext_df, household_golden_ext_df, cross_scale_golden_ext_df]:
    df["citation_required"] = df["user_query"].str.strip().map(
        lambda q: novelty_lookup.get(q, {}).get("citation_required", "no")
    )
    df["difficulty_aware"] = df["user_query"].str.strip().map(
        lambda q: novelty_lookup.get(q, {}).get("difficulty_aware", "no")
    )

# ── Add novelty columns to Phase 1 DFs with safe defaults ────────────────────
# Phase 1 queries are not pre-labelled — both default to 'no'
# All experiments handle them correctly via query.get() with 'no' defaults
for df in [gefcom_golden_df, household_golden_df, cross_scale_golden_df]:
    if "citation_required" not in df.columns:
        df["citation_required"] = "no"
    if "difficulty_aware" not in df.columns:
        df["difficulty_aware"] = "no"

# ── Extended column list (original 18 + 2 novelty) ───────────────────────────
extended_cols = GOLDEN_CSV_COLUMNS + ["citation_required", "difficulty_aware"]

# Ensure all DFs have all columns before concat
all_dfs = [
    gefcom_golden_df,
    household_golden_df,
    cross_scale_golden_df,
    gefcom_golden_ext_df,
    household_golden_ext_df,
    cross_scale_golden_ext_df,
]
for df in all_dfs:
    for col in extended_cols:
        if col not in df.columns:
            df[col] = ""

# ── Concatenate and deduplicate ───────────────────────────────────────────────
combined_200 = pd.concat(
    [df[extended_cols] for df in all_dfs],
    ignore_index=True,
)

before = len(combined_200)
combined_200 = combined_200.drop_duplicates(subset=["user_query"]).reset_index(drop=True)
if before != len(combined_200):
    print(f"Deduplication: removed {before - len(combined_200)} duplicate queries")

# ── Reassign clean sequential integer golden_ids (1 to N) ────────────────────
combined_200["golden_id"] = range(1, len(combined_200) + 1)

# ── Save ─────────────────────────────────────────────────────────────────────
output_path_200 = PATHS["golden_dataset"] / "combined_golden_dataset_200.csv"
combined_200.to_csv(output_path_200, index=False)

logger.info("Combined 200-query golden dataset saved: %d rows -> %s", len(combined_200), output_path_200)
print(f"Saved {len(combined_200)} rows to {output_path_200}")

2026-05-05 23:27:39 | INFO     | golden_dataset | Combined 200-query golden dataset saved: 200 rows -> E:\UpGrad\LJMU_Thesis\Topic_1_Energy_Forecasting\Research_Proposal\Git_Repo\RAG-Based-Energy-Forecasting\outputs\golden_dataset\combined_golden_dataset_200.csv



  STEP 4/7  [████░░░] 57%
  Phase 3 — Building combined 200-query golden dataset

Saved 200 rows to E:\UpGrad\LJMU_Thesis\Topic_1_Energy_Forecasting\Research_Proposal\Git_Repo\RAG-Based-Energy-Forecasting\outputs\golden_dataset\combined_golden_dataset_200.csv


---
## Cell 7 — Validation Report

In [7]:
log_section("Validation report", 5, 7)

print("\n" + "=" * 65)
print("  GOLDEN DATASET 200 — VALIDATION REPORT")
print("=" * 65)

if combined_200.empty:
    print("  Dataset is empty — check logs for errors.")
else:
    print(f"\n  Total queries : {len(combined_200)}  (target: 200)")
    print(f"  Columns       : {len(combined_200.columns)}")

    # Missing value check on critical columns
    critical_cols = [
        "user_query", "reference_answer", "expected_summary_ids",
        "expected_primary_summary_id", "expected_context_summary",
    ]
    print("\n  Missing value check:")
    for col in critical_cols:
        missing = combined_200[col].isna().sum()
        status  = "OK" if missing == 0 else f"WARNING — {missing} missing"
        print(f"    {col:<40} {status}")

    # Dataset source distribution
    print("\n  By dataset_source:")
    print(combined_200["dataset_source"].value_counts().to_string())

    # Query type distribution
    print("\n  By query_type:")
    print(combined_200["query_type"].value_counts().to_string())

    # Difficulty distribution
    print("\n  By difficulty_level:")
    print(combined_200["difficulty_level"].value_counts().to_string())

    # Retrieval strategy distribution
    print("\n  By retrieval_strategy_target:")
    print(combined_200["retrieval_strategy_target"].value_counts().to_string())

    # Novelty column distribution
    print("\n  Novelty column — citation_required:")
    print(combined_200["citation_required"].value_counts().to_string())

    print("\n  Novelty column — difficulty_aware:")
    print(combined_200["difficulty_aware"].value_counts().to_string())

    # Reference answer length
    combined_200["answer_words"] = combined_200["reference_answer"].str.split().str.len()
    print(f"\n  Reference answer length:"
          f" mean={combined_200['answer_words'].mean():.0f} words"
          f" | min={combined_200['answer_words'].min()}"
          f" | max={combined_200['answer_words'].max()}")

    # Sample entries
    print("\n" + "-" * 65)
    print("  SAMPLE ENTRIES (2 random)")
    print("-" * 65)
    for _, row in combined_200.sample(n=2, random_state=42).iterrows():
        print(f"\n  [{row['dataset_source']} / {row['query_type']} / "
              f"{row['difficulty_level']}]")
        print(f"  Query            : {row['user_query']}")
        answer_preview = str(row['reference_answer'])[:300]
        ellipsis = "..." if len(str(row['reference_answer'])) > 300 else ""
        print(f"  Answer           : {answer_preview}{ellipsis}")
        print(f"  citation_required: {row['citation_required']}")
        print(f"  difficulty_aware : {row['difficulty_aware']}")
        print(f"  Generated by     : {row['generated_by']} @ {row['generated_at']}")

print("\n" + "=" * 65)


  STEP 5/7  [█████░░] 71%
  Validation report


  GOLDEN DATASET 200 — VALIDATION REPORT

  Total queries : 200  (target: 200)
  Columns       : 20

  Missing value check:
    user_query                               OK
    reference_answer                         OK
    expected_summary_ids                     OK
    expected_primary_summary_id              OK
    expected_context_summary                 OK

  By dataset_source:
dataset_source
gefcom_ext         60
household_ext      54
cross_scale_ext    36
gefcom             20
household          18
cross_scale        12

  By query_type:
query_type
cross_scale      48
pattern          39
statistical      36
comparative      28
operational      24
appliance        13
zone_specific    12

  By difficulty_level:
difficulty_level
hard      104
medium     73
easy       23

  By retrieval_strategy_target:
retrieval_strategy_target
dense           86
hybrid          49
hierarchical    33
all             32

  Novelty column — citation_re

---
## Reload Cell — Run after kernel restart

If the kernel restarts at any point, run this single cell to restore all
variables from the saved CSV files without re-running Gemini generation.

In [ ]:
# ── RELOAD CELL — run this after any kernel restart ──────────────────────────
import pandas as pd
from config import PATHS

GOLDEN_DIR = PATHS["golden_dataset"]

# Load individual per-source files
_files = {
    "gefcom_golden_df":          "gefcom_golden_dataset.csv",
    "household_golden_df":       "household_golden_dataset.csv",
    "cross_scale_golden_df":     "cross_scale_golden_dataset.csv",
    "gefcom_golden_ext_df":      "gefcom_golden_ext.csv",
    "household_golden_ext_df":   "household_golden_ext.csv",
    "cross_scale_golden_ext_df": "cross_scale_golden_ext.csv",
}

for var_name, fname in _files.items():
    path = GOLDEN_DIR / fname
    if path.exists():
        globals()[var_name] = pd.read_csv(path)
        print(f"  {var_name}: {len(globals()[var_name])} rows loaded")
    else:
        print(f"  {var_name}: NOT FOUND at {path} — run generation cells first")

# Load the combined 200-query file
path_200 = GOLDEN_DIR / "combined_golden_dataset_200.csv"
if path_200.exists():
    combined_200 = pd.read_csv(path_200)
    print(f"\ncombined_200: {len(combined_200)} rows loaded")
    print(f"Columns: {combined_200.columns.tolist()}")
else:
    print(f"\ncombined_200: NOT FOUND — run Cell 6 (Phase 3) first")

---
## Pipeline Complete

Golden dataset files saved to `outputs/golden_dataset/`:

```
combined_golden_dataset_200.csv     <- load this in 08_experiments.ipynb
combined_golden_dataset.csv         <- original 50-query file (preserved as backup)
gefcom_golden_dataset.csv           <- Phase 1 GEFCom (20)
household_golden_dataset.csv        <- Phase 1 Household (18)
cross_scale_golden_dataset.csv      <- Phase 1 Cross-scale (12)
gefcom_golden_ext.csv               <- Phase 2 GEFCom (60)
household_golden_ext.csv            <- Phase 2 Household (54)
cross_scale_golden_ext.csv          <- Phase 2 Cross-scale (36)
```